In [1]:
import pandas as pd
import numpy as np
import os
from tqdm.auto import tqdm

In [2]:
# Create a schema
column_names = ['chromosome_name',
                'start',
                'end',
                'gene_id',
                'E066',
                'strand',
                'label',
                'external_gene_name',
                'start_position',
                'end_position',
                'tss' 
                ]

In [3]:
DATASET_PATH = '../dataset/E066/'

In [4]:
E066_df = pd.read_csv(os.path.join(DATASET_PATH, "E066.bed"), sep="\t", names = column_names)

In [5]:
E066_df.head()

,chromosome_name,start,end,gene_id,E066,strand,label,external_gene_name,start_position,end_position,tss
0,chrX,99889988,99899988,ENSG00000000003,73.205,-1,1,TSPAN6,99883667,99894988,99894988
1,chrX,99834799,99844799,ENSG00000000005,0.191,1,0,TNMD,99839799,99854882,99839799
2,chr20,49570092,49580092,ENSG00000000419,52.609,-1,1,DPM1,49551404,49575092,49575092
3,chr1,169858408,169868408,ENSG00000000457,4.733,-1,1,SCYL3,169818772,169863408,169863408
4,chr1,169626245,169636245,ENSG00000000460,0.942,1,0,C1orf112,169631245,169823221,169631245


# Reformating the histone

In [6]:
histone_list = ['H3K4me1', 'H3K4me3', 'H3K9me3', 'H3K27me3', 'H3K36me3']

In [7]:
intersect_cols = [  'chromosome_name',
                    'start',
                    'end',
                    'gene_id',
                    'E066',
                    'strand',
                    'label',
                    'external_gene_name',
                    'start_position',
                    'end_position',
                    'tss',
                    'chrom',
                    'chromStart',
                    'chromEnd',
                    'name',
                    'score',
                    'strand_peak',
                    'signalValue',
                    'pValue',
                    'qValue',
                    'peak'
                ]

In [8]:
for name in histone_list:
    print(f"Processing: {name}")
    histone_df = pd.read_csv(os.path.join(DATASET_PATH, "intersect", f"E066_{name}.bed"), sep='\t', names=intersect_cols)
    histone_df.loc[:, 'startBucket'] = (histone_df['chromStart'] - histone_df['start'])/100
    histone_df.loc[:, 'endBucket'] = (histone_df['chromEnd'] - histone_df['start'])/100
    histone_df.to_csv(os.path.join(DATASET_PATH, f"E066_{name}_df.csv"), header=True, index=False)
    print(f"CSV saved!")

Processing: H3K4me1
CSV saved!
Processing: H3K4me3
CSV saved!
Processing: H3K9me3
CSV saved!
Processing: H3K27me3
CSV saved!
Processing: H3K36me3
CSV saved!


# Reading Histone File

In [ ]:
H3K4me1_df = pd.read_csv(os.path.join(DATASET_PATH, "E066_H3K4me1_df.csv"))
H3K4me3_df = pd.read_csv(os.path.join(DATASET_PATH, "E066_H3K4me3_df.csv"))
H3K9me3_df = pd.read_csv(os.path.join(DATASET_PATH, "E066_H3K9me3_df.csv"))
H3K27me3_df = pd.read_csv(os.path.join(DATASET_PATH, "E066_H3K27me3_df.csv"))
H3K36me3_df = pd.read_csv(os.path.join(DATASET_PATH, "E066_H3K36me3_df.csv"))

In [ ]:
print(H3K4me1_df.shape)
print(H3K4me3_df.shape)
print(H3K9me3_df.shape)
print(H3K27me3_df.shape)
print(H3K36me3_df.shape)

In [ ]:
H3K4me1_df.head(10)

In [ ]:
H3K36me3_df.head(10)

# Function to encode the gene and histone

In [ ]:
# Load the gene expression data

gene_exp_arr = E066_df.to_numpy()
display(gene_exp_arr)

In [ ]:
gene_exp_arr.shape[0]

In [ ]:
def encode_histone_exp(gene_id, start, end, histone):
    arr = []
    count = 1
    for i in range(start, end, 100):
        histone_df = histone[
                    # The same gene ID
                    (histone['gene_id'] == gene_id) &
                    (
                        # The window is inside the histone
                        ((histone['chromStart'] <= i) & (histone['chromEnd'] >= i + 100)) |
                        # The start of histone is inside the window
                        ((histone['chromStart'] >= i) & (histone['chromStart'] <= i + 100) & (histone['chromEnd'] >= i + 100)) |
                        # The end of histone is inside the window
                        ((histone['chromStart'] <= i) & (histone['chromEnd'] >= i) & (histone['chromEnd'] <= i + 100))
                    )
                    ]
        size = histone_df.shape[0]
        if (size > 0):
            avg = histone_df['signalValue'].mean()
        else:
            avg = 0.0
        
        # print(f"Bin {count}: {i} to {i + 100}: {histone_df.shape[0]}, average: {avg}")
        arr.append(avg)
        count += 1
    return arr

In [ ]:
# Use this function for apply()
def encode_histone_exp(row, histone):
    arr = []
    start = row['start']
    end = row['end']
    gene_id = row['gene_id']
    count = 1

    for i in range(start, end, 100):
        histone_df = histone[
                    # The same gene ID
                    (histone['gene_id'] == gene_id) &
                    (
                        # The window is inside the histone
                        ((histone['chromStart'] <= i) & (histone['chromEnd'] >= i + 100)) |
                        # The start of histone is inside the window
                        ((histone['chromStart'] >= i) & (histone['chromStart'] <= i + 100) & (histone['chromEnd'] >= i + 100)) |
                        # The end of histone is inside the window
                        ((histone['chromStart'] <= i) & (histone['chromEnd'] >= i) & (histone['chromEnd'] <= i + 100))
                    )
                    ]
        size = histone_df.shape[0]
        if (size > 0):
            avg = histone_df['signalValue'].mean()
        else:
            avg = 0.0
        
        # print(f"Bin {count}: {i} to {i + 100}: {histone_df.shape[0]}, average: {avg}")
        arr.append(avg)
        count += 1
    return arr

In [ ]:
H3K4me1_stack = np.empty((0, 100))

In [ ]:
samples = gene_exp_arr[0:10, :]

In [ ]:
samples

In [ ]:
count = 1
for row in tqdm(samples, "Processing"):
    gene_id = row[3]
    tss_start = row[1]
    tss_end = row[2]

    print(f"{count} -  {gene_id}")
    arr = encode_histone_exp(gene_id, tss_start, tss_end, E066_H3K4me1_df)
    H3K4me1_stack = np.vstack((H3K4me1_stack, arr))
    count += 1

In [ ]:
E066_10_df = E066_df.head(10)
E066_10_df

In [ ]:
tqdm.pandas()

In [ ]:
E066_10_df.loc[:, 'H3K4me1'] = E066_10_df.apply(lambda row: encode_histone_exp(row, H3K4me1_df), axis = 1)

In [ ]:
E066_10_df.to_parquet(os.path.join(DATASET_PATH, "test.parquet"))

In [ ]:
test_df = pd.read_parquet(os.path.join(DATASET_PATH, "test.parquet"))